Instead of:

User → LLM → Answer

We do:

User → LLM decides → Call Tool → Get Result → LLM reasons → Final Answer

The LLM becomes a decision-maker, not just a text generator.

In [19]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [20]:
from langchain.tools import tool
from duckduckgo_search import DDGS

@tool
def search_tool(query: str) -> str:
    """Search the web for current information."""
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=3)
        return "\n".join([r["body"] for r in results])

In [21]:
llm_with_tools = llm.bind_tools([search_tool])

In [22]:
from typing import TypedDict
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages: list[BaseMessage]

In [23]:
def llm_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": state["messages"] + [response]}

In [24]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode([search_tool])

In [25]:
from langgraph.graph import END

def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

In [26]:
from langgraph.graph import StateGraph

workflow = StateGraph(AgentState)

workflow.add_node("llm", llm_node)
workflow.add_node("tools", tool_node)

workflow.set_entry_point("llm")

workflow.add_conditional_edges(
    "llm",
    should_continue,
    {
        "tools": "tools",
        END: END,
    }
)

workflow.add_edge("tools", "llm")

graph = workflow.compile()

In [28]:
from langchain_core.messages import HumanMessage

result = graph.invoke({
    "messages": [HumanMessage(content="Who is the current Prime Minister of India?")]
})

print(result["messages"][-1].content)

APIConnectionError: Connection error.